In [1]:
import nflreadpy
import pandas as pd
from pathlib import Path

# Create PickleFiles directory in Models/ (one level up from this notebook)
Path("../PickleFiles").mkdir(exist_ok=True)

years = list(range(2014, 2026))

# --- Get team points for/against from schedules ---
schedules = nflreadpy.load_schedules(years).to_pandas()
schedules = schedules[(schedules['game_type'] == 'REG') & (schedules['home_score'].notna())].copy()

# Home games
home = schedules.groupby(['season', 'home_team']).agg(
    pts_for=('home_score', 'sum'),
    pts_against=('away_score', 'sum'),
    games=('home_score', 'count')
).reset_index().rename(columns={'home_team': 'team'})

# Away games
away = schedules.groupby(['season', 'away_team']).agg(
    pts_for=('away_score', 'sum'),
    pts_against=('home_score', 'sum'),
    games=('away_score', 'count')
).reset_index().rename(columns={'away_team': 'team'})

team_pts = pd.concat([home, away]).groupby(['season', 'team']).sum().reset_index()
team_pts['offensive_points_per_game'] = team_pts['pts_for'] / team_pts['games']
team_pts['defensive_points_per_game'] = team_pts['pts_against'] / team_pts['games']

# --- Get team rushing/passing yards from weekly data ---
# nflreadpy weekly uses 'team' (not 'recent_team')
weekly = nflreadpy.load_player_stats(years, summary_level='week').to_pandas()[
    ['player_id', 'team', 'season', 'week', 'rushing_yards', 'passing_yards']
]

team_yards = weekly.groupby(['season', 'team']).agg(
    team_rush_yards=('rushing_yards', 'sum'),
    team_passing_yards=('passing_yards', 'sum')
).reset_index()
team_yards['team_total_yards'] = team_yards['team_rush_yards'] + team_yards['team_passing_yards']

# --- Merge into one DataFrame ---
AVgrades = pd.merge(team_pts, team_yards, on=['season', 'team'], how='inner')

# Teams use NFL abbreviations (ARI, BAL, etc.) from nflverse

AVgrades = AVgrades.rename(columns={
    'pts_for': 'total_offense_points',
    'season': 'year'
})

print(f"AVgrades shape: {AVgrades.shape}")
print(f"Years: {sorted(AVgrades['year'].unique())}")
print(f"Teams per year: {AVgrades.groupby('year')['team'].nunique().unique()}")
AVgrades.head()


AVgrades shape: (373, 10)
Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Teams per year: [29 30 31 32]


,year,team,total_offense_points,pts_against,games,offensive_points_per_game,defensive_points_per_game,team_rush_yards,team_passing_yards,team_total_yards
0,2014,ARI,310,299,16,19.3750,18.6875,1335,4072,5407
1,2014,ATL,381,417,16,23.8125,26.0625,1498,4758,6256
2,2014,BAL,409,302,16,25.5625,18.8750,2208,4537,6745
3,2014,BUF,343,289,16,21.4375,18.0625,1482,3856,5338
4,2014,CAR,339,374,16,21.1875,23.3750,2356,4272,6628


In [2]:
AVgrades = AVgrades.sort_values(by=['year','team'])

# Using points per game as proxy for points per drive (ratio to league avg is nearly identical)
AVgrades['league_avg_offensive_ppg'] = AVgrades.groupby('year')['offensive_points_per_game'].transform('mean')
AVgrades['league_avg_defensive_ppg'] = AVgrades.groupby('year')['defensive_points_per_game'].transform('mean')

AVgrades["team_offense_points"] = (100 * AVgrades['offensive_points_per_game']) / AVgrades['league_avg_offensive_ppg']
AVgrades["team_points_for_o_line"] = (5/11) * AVgrades["team_offense_points"]
AVgrades["team_points_for_skill_positions"] = AVgrades["team_offense_points"] - AVgrades["team_points_for_o_line"]
AVgrades["team_points_for_rushers"] = AVgrades["team_points_for_skill_positions"] * 0.22 * (AVgrades["team_rush_yards"]/AVgrades["team_total_yards"]) / 0.37
AVgrades["team_points_for_passers"] = (AVgrades["team_points_for_skill_positions"] - AVgrades["team_points_for_rushers"]) * 0.26
AVgrades["team_points_for_receivers"] = (AVgrades["team_points_for_skill_positions"] - AVgrades["team_points_for_rushers"]) * 0.74
AVgrades["M"] = AVgrades["defensive_points_per_game"] / AVgrades['league_avg_defensive_ppg']
AVgrades["team_defense_points"] = 100 * ((1+2*AVgrades["M"]-AVgrades["M"]**2) / (2*AVgrades["M"]))

AVgrades

,year,team,total_offense_points,pts_against,games,offensive_points_per_game,defensive_points_per_game,team_rush_yards,team_passing_yards,team_total_yards,league_avg_offensive_ppg,league_avg_defensive_ppg,team_offense_points,team_points_for_o_line,team_points_for_skill_positions,team_points_for_rushers,team_points_for_passers,team_points_for_receivers,M,team_defense_points
0,2014,ARI,310,299,16,19.375000,18.687500,1335,4072,5407,22.931034,22.437500,84.492481,38.405673,46.086808,6.765852,10.223449,29.097508,0.832869,118.389991
1,2014,ATL,381,417,16,23.812500,26.062500,1498,4758,6256,22.931034,22.437500,103.843985,47.201811,56.642174,8.064472,12.630202,35.947499,1.161560,84.967569
2,2014,BAL,409,302,16,25.562500,18.875000,2208,4537,6745,22.931034,22.437500,111.475564,50.670711,60.804853,11.835219,12.732105,36.237529,0.841226,117.375805
3,2014,BUF,343,289,16,21.437500,18.062500,1482,3856,5338,22.931034,22.437500,93.486842,42.494019,50.992823,8.417820,11.069501,31.505502,0.805014,121.860030
4,2014,CAR,339,374,16,21.187500,23.375000,2356,4272,6628,22.931034,22.437500,92.396617,41.998462,50.398154,10.651932,10.334018,29.412205,1.041783,95.905516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368,2025,SEA,483,292,17,28.411765,17.176471,2487,4735,7222,23.012868,23.012868,123.460340,56.118336,67.342004,13.788763,13.923843,39.629399,0.746385,129.670237
369,2025,SF,437,371,17,25.705882,21.823529,2000,4753,6753,23.012868,23.012868,111.702213,50.773733,60.928480,10.729378,13.051766,37.147335,0.948319,105.308971
370,2025,TB,380,411,17,22.352941,24.176471,1947,3755,5702,23.012868,23.012868,97.132359,44.151072,52.981287,10.756778,10.978372,31.246137,1.050563,95.065365
371,2025,TEN,284,478,17,16.705882,28.117647,1589,3241,4830,23.012868,23.012868,72.593658,32.997117,39.596541,7.745598,8.281245,23.569697,1.221823,79.831322


In [3]:
AVgrades["team_receiving_yards"] = AVgrades["team_passing_yards"]

In [4]:
AVgrades = AVgrades.rename(columns={'team_points_for_o_line': 'oline', 'team_points_for_rushers': 'rb', 'team_points_for_passers': 'qb', 'team_points_for_receivers':'wrte', 'team_defense_points':'dst', 'year':'season'})
AVgrades.to_pickle("../PickleFiles/AVgrades.pkl")

In [5]:
AVgrades1 = AVgrades[['team','oline','qb','rb','wrte','dst','season']]
AVgrades1.to_pickle("../PickleFiles/AVbyPositionGroup.pkl")